# Character Model (CM) Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enph-479-edge-ai-stt/enph-479-edge-ai-stt/blob/main/training/notebooks/cm_training.ipynb)

Trains the character-level language model (2x256 unidirectional LSTM) on a random sample of the LibriSpeech LM text and prints dev-clean bits per character after every epoch, then fine-tunes it with its weights quantized to 6 bits. **Runtime: T4 GPU.** Runs start to finish in one session, then saves both models, their training logs and the 6-bit model's weight image for the FPGA to a run folder on Google Drive.

Follows the paper's recipe: randomly selected sentences concatenated into one stream with EOS between them, one-hot character in, next-character distribution out, truncated BPTT with the LSTM state carried across chunks. The LM shares the AM's 30-symbol vocab (provisional until `shared/specs` is frozen); the CTC blank never occurs in text. One departure: Adam instead of the paper's AdaDelta, like the AM, since AdaDelta converged much more slowly.

## 1. Pull the repo

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/enph-479-edge-ai-stt/enph-479-edge-ai-stt.git"
REPO_DIR = "/content/enph-479-edge-ai-stt"
BRANCH = "feat/drive-data-cache"  # set to a feature branch to test it before merging

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
# Always land on the tip of BRANCH, even if an earlier run left the clone on another branch.
git = ["git", "-C", REPO_DIR]
subprocess.run([*git, "fetch", "origin", BRANCH], check=True)
subprocess.run([*git, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run([*git, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Installs this checkout's package and its dependencies (jiwer, soundfile);
# torch and torchaudio come with Colab.
training_dir = os.path.join(REPO_DIR, "training")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", training_dir], check=True)

# The editable install only takes effect at interpreter start, so put src/ on the
# path for this already-running kernel.
sys.path.insert(0, os.path.join(training_dir, "src"))
importlib.invalidate_caches()

## 2. Mount Drive and configure the run

Mounting Drive here means the permission prompt comes up now, not after training. The results go to `MyDrive/enph479-stt/runs/<start time>-cm/` at the end. The downloaded LibriSpeech archives are kept in `MyDrive/enph479-stt/data/`.

`keep` is the fraction of the corpus's 40.4M sentences (4.25B characters) to train on. 0.05 gives about 2M sentences, 210M characters, roughly the size of the WSJ LM text the paper used. Not yet timed on a T4: check ms/batch in the first progress lines and adjust `epochs` to fit the session. The 6-bit fine-tune adds 1 more epoch.

In [ ]:
import time

from google.colab import drive

from training.cm.train import TrainConfig, load_data, train

drive.mount("/content/drive")
RUN_DIR = f"/content/drive/MyDrive/enph479-stt/runs/{time.strftime('%Y%m%d-%H%M')}-cm"
DATA_CACHE = "/content/drive/MyDrive/enph479-stt/data"

cfg = TrainConfig(epochs=5, keep=0.05)
print(RUN_DIR)
cfg

## 3. Data

Puts the normalized LM text (1.5 GB gzip, kept gzipped) and dev-clean (337 MB, for its transcripts only) in `/content/data`. Streams the corpus once to sample `cfg.keep` of its sentences, then encodes both sides to EOS-joined index streams. The LM text excludes the dev and test books, so dev-clean is held out.

Each archive is copied from `DATA_CACHE` on Drive if it is there. If not, it is downloaded from OpenSLR and copied into `DATA_CACHE`, so only the first run downloads. Either way it is md5-checked on the Colab disk. The AM notebook uses the same cache, so dev-clean is downloaded once for both.

In [ ]:
streams = load_data("/content/data", DATA_CACHE, cfg)

## 4. Train

Bits per character starts near log2(30) = 4.9 (uniform), drops quickly past the unigram level (~4.1) as the model picks up spelling, then improves slowly as it picks up words. Each epoch line also prints a sampled line of text, which should read as more and more English-like. Progress lines show ms/batch (a batch is one 100-character chunk of every stream); each epoch line shows train and eval time.

In [ ]:
model = train(cfg, streams, "train.log")

## 5. Fine-tune with 6-bit weights

Starts from the float model above and trains 1 more epoch at a tenth of the learning rate with every weight matrix quantized to 6 bits (63 levels) on the FPGA's fixed-point grid, the paper's retraining step and the same one the AM gets. The bits per character printed here is the 6-bit model's. Activations and the cell state are still float.

In [ ]:
from dataclasses import replace

qcfg = replace(cfg, epochs=1, lr=2e-4, quantize=True)
qmodel = train(qcfg, streams, "train_6bit.log", model.state_dict())

## 6. Save to Drive

Writes `cm.pt` and `cm_6bit.pt` (the weights) and both training logs to the run folder. Also writes the 6-bit model packed for the FPGA: `cm_fabric.mem` is the contents of the fabric's weight memory in the order the board streams it in, and `cm_fabric.json` is its dimensions and input scale (layout in `shared/specs/weight_image.md`). Then flushes Drive so the upload finishes before the session ends.

In [ ]:
from training.lstm.export import save_run

save_run(RUN_DIR, "cm", model, qmodel)
print(f"saved both models, both training logs and the FPGA weight image to {RUN_DIR}")
drive.flush_and_unmount()